In [60]:
import sqlite3
import json
import sys
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [61]:
def extract_parameters(json_str):
    if json_str is None:
        return None
    try:
        data_dict = json.loads(json_str)
        return data_dict.get("parameters")
    except json.JSONDecodeError:
        return None

In [62]:
db_path = '/home3/p302242/nni-experiments/dwpxyan5/db/nni.sqlite'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

1. List all tables in the database

In [63]:

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Tables in the database:")
for t in tables:
    print("-", t[0])


Tables in the database:
- TrialJobEvent
- MetricData
- ExperimentProfile


2. Show the schema (columns) of each table

In [64]:
for t in tables:
    print(f"\nSchema of table {t[0]}:")
    cursor.execute(f"PRAGMA table_info({t[0]});")
    columns = cursor.fetchall()
    for col in columns:
        _ = ''
        for i in col:
            _ += f' {i} '
        print(_)


Schema of table TrialJobEvent:
 0  timestamp  integer  0  None  0 
 1  trialJobId  text  0  None  0 
 2  event  text  0  None  0 
 3  data  text  0  None  0 
 4  logPath  text  0  None  0 
 5  sequenceId  integer  0  None  0 
 6  message  text  0  None  0 
 7  environmentId  text  0  None  0 

Schema of table MetricData:
 0  timestamp  integer  0  None  0 
 1  trialJobId  text  0  None  0 
 2  parameterId  text  0  None  0 
 3  type  text  0  None  0 
 4  sequence  integer  0  None  0 
 5  data  text  0  None  0 

Schema of table ExperimentProfile:
 0  params  text  0  None  0 
 1  id  text  0  None  0 
 2  execDuration  integer  0  None  0 
 3  startTime  integer  0  None  0 
 4  endTime  integer  0  None  0 
 5  logDir  text  0  None  0 
 6  nextSequenceId  integer  0  None  0 
 7  revision  integer  0  None  0 


3. See first few rows of a table

In [65]:
table_name = 'TrialJobEvent'
cursor.execute(f"SELECT * FROM {table_name} LIMIT 5;")
rows = cursor.fetchall()
print(f"\nFirst 5 rows of {table_name}:")
for row in rows:
    print(row)


First 5 rows of TrialJobEvent:
(1767613690409, 'M1URt', 'RUNNING', '{"parameter_id": 0, "parameter_source": "algorithm", "parameters": {"intermediate_dim": 64, "embedding_dim": 64, "num_attention_heads": 2, "training_seq_len": 12, "inference_seq_len": 12, "lr": 0.000722027266336143, "hidden_dropout_prob": 0.23950932870044567, "epochs": 80}, "parameter_index": 0}', '/home3/p302242/nni-experiments/dwpxyan5/environments/local-env/trials/M1URt', 0, None, 'local-env')
(1767613690471, 'SMNWG', 'RUNNING', '{"parameter_id": 1, "parameter_source": "algorithm", "parameters": {"intermediate_dim": 128, "embedding_dim": 64, "num_attention_heads": 2, "training_seq_len": 16, "inference_seq_len": 16, "lr": 0.00012127018757407778, "hidden_dropout_prob": 0.2046560000161886, "epochs": 90}, "parameter_index": 0}', '/home3/p302242/nni-experiments/dwpxyan5/environments/local-env/trials/SMNWG', 1, None, 'local-env')
(1767617649632, 'M1URt', 'SUCCEEDED', None, '/home3/p302242/nni-experiments/dwpxyan5/environ

In [66]:
table_name = 'MetricData'
cursor.execute(f"SELECT * FROM {table_name} LIMIT 5;")
rows = cursor.fetchall()
print(f"\nFirst 5 rows of {table_name}:")
for row in rows:
    print(row)


First 5 rows of MetricData:
(1767617648789, 'M1URt', '0', 'FINAL', 0, '"0.23728716580424347"')
(1767618136882, 'SMNWG', '1', 'FINAL', 0, '"-0.0668618100583333"')
(1767620626077, 'a05Wo', '2', 'FINAL', 0, '"0.08069532863015844"')
(1767621992094, 'F5Hc0', '3', 'FINAL', 0, '"0.23846875554412567"')
(1767624743577, 'WD4qG', '4', 'FINAL', 0, '"0.12857481427813724"')


In [67]:
table_name = 'ExperimentProfile'
cursor.execute(f"SELECT * FROM {table_name} LIMIT 5;")
rows = cursor.fetchall()
print(f"\nFirst 5 rows of {table_name}:")
for row in rows:
    print(row)


First 5 rows of ExperimentProfile:
('{"experimentName":"HPO_lander_baseline","experimentType":"hpo","searchSpace":{"intermediate_dim":{"_type":"choice","_value":[64,128]},"embedding_dim":{"_type":"choice","_value":[32,64]},"num_attention_heads":{"_type":"choice","_value":[1,2]},"training_seq_len":{"_type":"choice","_value":[12,14,16]},"inference_seq_len":{"_type":"choice","_value":[12,14,16]},"lr":{"_type":"uniform","_value":[0.0001,0.0009]},"hidden_dropout_prob":{"_type":"uniform","_value":[0.15,0.275]},"epochs":{"_type":"choice","_value":[60,70,80,90]}},"trialCommand":"/home3/p302242/venvs/nni-venv/bin/python3 -m src.hpo.nni_optmin --config_json /scratch/p302242/lunar-transformer-il/configs/nni_config.json","trialCodeDirectory":"/scratch/p302242/lunar-transformer-il","trialConcurrency":2,"maxTrialNumber":200,"useAnnotation":false,"debug":false,"logLevel":"info","experimentWorkingDirectory":"/home3/p302242/nni-experiments","tuner":{"name":"TPE","classArgs":{"optimize_mode":"maximize"

# Extracting Fitness Scores

In [68]:
df_score = pd.read_sql("SELECT trialJobId, data FROM MetricData", conn)
df_score

,trialJobId,data
0,M1URt,"""0.23728716580424347"""
1,SMNWG,"""-0.0668618100583333"""
2,a05Wo,"""0.08069532863015844"""
3,F5Hc0,"""0.23846875554412567"""
4,WD4qG,"""0.12857481427813724"""
...,...,...
156,hTrOK,"""0.339875298696346"""
157,fkIbl,"""0.32228560129241296"""
158,Rc0W2,"""0.10045044693351782"""
159,lDzoY,"""-0.1205793373586532"""


# Extracting Trial Parameters

In [69]:
df_params = pd.read_sql("SELECT trialJobId, data FROM TrialJobEvent", conn)
df_params

,trialJobId,data
0,M1URt,"{""parameter_id"": 0, ""parameter_source"": ""algor..."
1,SMNWG,"{""parameter_id"": 1, ""parameter_source"": ""algor..."
2,M1URt,None
3,a05Wo,"{""parameter_id"": 2, ""parameter_source"": ""algor..."
4,SMNWG,None
...,...,...
319,mzFuc,"{""parameter_id"": 160, ""parameter_source"": ""alg..."
320,lDzoY,None
321,T1c4I,"{""parameter_id"": 161, ""parameter_source"": ""alg..."
322,mzFuc,None


In [70]:
df_params['parameters'] = df_params['data'].apply(extract_parameters)

In [71]:
df_params = pd.concat([df_params.drop(columns=['data', 'parameters']), df_params['parameters'].apply(pd.Series)], axis=1).drop(columns=['num_attention_heads'])

In [72]:
df_params

,trialJobId,intermediate_dim,embedding_dim,training_seq_len,inference_seq_len,lr,hidden_dropout_prob,epochs
0,M1URt,64.0,64.0,12.0,12.0,0.000722,0.239509,80.0
1,SMNWG,128.0,64.0,16.0,16.0,0.000121,0.204656,90.0
2,M1URt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,a05Wo,128.0,32.0,12.0,16.0,0.000764,0.272913,60.0
4,SMNWG,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
319,mzFuc,128.0,64.0,16.0,12.0,0.000207,0.206110,60.0
320,lDzoY,NaN,NaN,NaN,NaN,NaN,NaN,NaN
321,T1c4I,128.0,64.0,16.0,12.0,0.000211,0.206807,60.0
322,mzFuc,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Combine Score and Parameters

In [73]:
df = pd.merge(df_score, df_params, on='trialJobId', how='inner').dropna() # 'inner' keeps only matching trialJobIds, `.dropna` removes rows where a NaN appears
df

,trialJobId,data,intermediate_dim,embedding_dim,training_seq_len,inference_seq_len,lr,hidden_dropout_prob,epochs
0,M1URt,"""0.23728716580424347""",64.0,64.0,12.0,12.0,0.000722,0.239509,80.0
2,SMNWG,"""-0.0668618100583333""",128.0,64.0,16.0,16.0,0.000121,0.204656,90.0
4,a05Wo,"""0.08069532863015844""",128.0,32.0,12.0,16.0,0.000764,0.272913,60.0
6,F5Hc0,"""0.23846875554412567""",64.0,64.0,12.0,16.0,0.000339,0.242594,80.0
8,WD4qG,"""0.12857481427813724""",128.0,32.0,16.0,16.0,0.000889,0.260358,90.0
...,...,...,...,...,...,...,...,...,...
312,hTrOK,"""0.339875298696346""",128.0,64.0,16.0,12.0,0.000438,0.230254,60.0
314,fkIbl,"""0.32228560129241296""",128.0,64.0,16.0,12.0,0.000372,0.224515,60.0
316,Rc0W2,"""0.10045044693351782""",128.0,64.0,16.0,12.0,0.000244,0.222641,60.0
318,lDzoY,"""-0.1205793373586532""",128.0,64.0,16.0,12.0,0.000157,0.219357,60.0


In [74]:
df['trialJobId'] = df['trialJobId'].astype('string')
df['data'] = df['data'] = df['data'].str.replace(r'^"|"$', '', regex=True).astype(float)
df['training_seq_len'] = df['training_seq_len'].astype('Int64')
df['inference_seq_len'] = df['inference_seq_len'].astype('Int64')       
df['lr'] = df['lr'].astype('float')
df['hidden_dropout_prob'] = df['hidden_dropout_prob'].astype('float')
df['epochs'] = df['epochs'].astype('Int64')
df


,trialJobId,data,intermediate_dim,embedding_dim,training_seq_len,inference_seq_len,lr,hidden_dropout_prob,epochs
0,M1URt,0.237287,64.0,64.0,12,12,0.000722,0.239509,80
2,SMNWG,-0.066862,128.0,64.0,16,16,0.000121,0.204656,90
4,a05Wo,0.080695,128.0,32.0,12,16,0.000764,0.272913,60
6,F5Hc0,0.238469,64.0,64.0,12,16,0.000339,0.242594,80
8,WD4qG,0.128575,128.0,32.0,16,16,0.000889,0.260358,90
...,...,...,...,...,...,...,...,...,...
312,hTrOK,0.339875,128.0,64.0,16,12,0.000438,0.230254,60
314,fkIbl,0.322286,128.0,64.0,16,12,0.000372,0.224515,60
316,Rc0W2,0.100450,128.0,64.0,16,12,0.000244,0.222641,60
318,lDzoY,-0.120579,128.0,64.0,16,12,0.000157,0.219357,60


In [75]:
df.loc[df['data'].idxmax()]

trialJobId                M0gRz
data                   0.375748
intermediate_dim          128.0
embedding_dim              64.0
training_seq_len             14
inference_seq_len            12
lr                     0.000793
hidden_dropout_prob    0.157674
epochs                       60
Name: 170, dtype: object

# Plotting

In [76]:
clumns_2_plot = reversed(list(df))

In [77]:
threshold = df['data'].quantile(0.9)
top_10_percent = df[df['data'] >= threshold].sort_values(by='data', ascending=False)
top_10_percent

,trialJobId,data,intermediate_dim,embedding_dim,training_seq_len,inference_seq_len,lr,hidden_dropout_prob,epochs
170,M0gRz,0.375748,128.0,64.0,14,12,0.000793,0.157674,60
282,yCEYH,0.370861,128.0,64.0,16,12,0.000452,0.218557,60
158,ke8Gv,0.365268,128.0,64.0,14,12,0.000649,0.197817,60
306,VJjmG,0.347187,128.0,64.0,16,12,0.000409,0.226872,60
18,cLclb,0.346143,128.0,64.0,12,14,0.000577,0.224336,60
186,amZLO,0.342690,128.0,64.0,14,12,0.000749,0.151879,60
162,Z3dOb,0.342612,128.0,64.0,14,12,0.000693,0.182125,60
296,oOKtz,0.339934,128.0,64.0,16,12,0.000375,0.230192,60
312,hTrOK,0.339875,128.0,64.0,16,12,0.000438,0.230254,60
276,YyVya,0.336829,128.0,64.0,16,12,0.000387,0.189099,60


Plotly parallel coordinates plot

In [78]:
plt.figure(figsize=(12,8))

fig = px.parallel_coordinates(
    top_10_percent,
    dimensions=clumns_2_plot,
    color='data',  # Color lines by fitness value
    color_continuous_scale=px.colors.diverging.Tealrose,
    color_continuous_midpoint=top_10_percent['data'].mean()
)

# fig.update_traces()

fig.update_layout(
    plot_bgcolor='white'
)

fig.show()

<Figure size 1200x800 with 0 Axes>